# RAG Pipelines- Data ingestion to VectorDB pipeline

In [7]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_core import documents
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [18]:
# Read all the Pdfs in the directory
def process_all_pdfs(pdf_directory):
    """" reads all pdf files in pdf_directory """
    all_docs= []
    pdf_dir = Path(pdf_directory)

    # finds all pdfs recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}")
        try:
            #load the PDF page by page
            loader = PyMuPDFLoader(pdf_file)
            pages = loader.load()

            #Tag each page so we know exactly which file it came from earlier
            for page in pages:
                page.metadata["source"] = pdf_file.name
            all_docs.extend(pages)

        except Exception as e:
            print(f"Error loading {pdf_file.name} : {e}")
    print(f"\nProcessed {len(all_docs)} PDF files")
    return all_docs

#process all PDFs ion the data directory
alldocs = process_all_pdfs("../data")






Found 3 PDF files to process
Processing ../data/pdf/doc_01_rag_and_langchain.pdf
Processing ../data/pdf/doc_02_vector_databases.pdf
Processing ../data/pdf/doc_03_prompting_and_guardrails.pdf

Processed 3 PDF files


In [6]:
print(alldocs)

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-03-28T15:59:08+05:00', 'source': 'doc_01_rag_and_langchain.pdf', 'file_path': '../data/pdf/doc_01_rag_and_langchain.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': 'RAG + LangChain: Practical Notes', 'author': 'anonymous', 'subject': 'unspecified', 'keywords': '', 'moddate': '2026-03-28T15:59:08+05:00', 'trapped': '', 'modDate': "D:20260328155908+05'00'", 'creationDate': "D:20260328155908+05'00'", 'page': 0}, page_content="RAG + LangChain: Practical Notes\nRetrieval-Augmented Generation (RAG) improves LLM answers by grounding generation in\nexternal documents. Instead of relying only on the model's parameters, a retriever selects\nrelevant chunks from a knowledge base and passes them to the LLM as context.\nIn LangChain, a typical pipeline has: (1) document loading, (2) text splitting, (3) embeddings, (4)\nvector store indexing, and (5) a retrieval + generation ch

In [15]:
### Test splitting into chunks

def document_split(documents, chunk_size = 1000, chunk_overlap=200):
    """ split documents into smaller chunks for better rag performance """

    # This object defines the rules for how we cut the text
    text_splitter = RecursiveCharacterTextSplitter(
        #the maximum number of character in one chunk
        chunk_size = chunk_size,
        #to keep context connected
        chunk_overlap = chunk_overlap,

        length_function = len,

        # priority list of where to cut
        separators = ["\n\n", "\n", " ", ""]
    )
    #This takes the list of PDF pages and returns a much longer list of chunks.
    split_docs = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #show preview of first chunk
    if split_docs:
        print(f"\n----Preview of first chunk 0 -----")
        print(f"Content: {split_docs[0].page_content[:200]}")
        print(f"Metadta: {split_docs[0].metadata}")

    return split_docs

In [16]:
chunks = document_split(alldocs)
chunks

Split 3 documents into 4 chunks

----Preview of first chunk 0 -----
Content: RAG + LangChain: Practical Notes
Retrieval-Augmented Generation (RAG) improves LLM answers by grounding generation in
external documents. Instead of relying only on the model's parameters, a retriever
Metadta: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-03-28T15:59:08+05:00', 'source': 'doc_01_rag_and_langchain.pdf', 'file_path': '../data/pdf/doc_01_rag_and_langchain.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': 'RAG + LangChain: Practical Notes', 'author': 'anonymous', 'subject': 'unspecified', 'keywords': '', 'moddate': '2026-03-28T15:59:08+05:00', 'trapped': '', 'modDate': "D:20260328155908+05'00'", 'creationDate': "D:20260328155908+05'00'", 'page': 0}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-03-28T15:59:08+05:00', 'source': 'doc_01_rag_and_langchain.pdf', 'file_path': '../data/pdf/doc_01_rag_and_langchain.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': 'RAG + LangChain: Practical Notes', 'author': 'anonymous', 'subject': 'unspecified', 'keywords': '', 'moddate': '2026-03-28T15:59:08+05:00', 'trapped': '', 'modDate': "D:20260328155908+05'00'", 'creationDate': "D:20260328155908+05'00'", 'page': 0}, page_content="RAG + LangChain: Practical Notes\nRetrieval-Augmented Generation (RAG) improves LLM answers by grounding generation in\nexternal documents. Instead of relying only on the model's parameters, a retriever selects\nrelevant chunks from a knowledge base and passes them to the LLM as context.\nIn LangChain, a typical pipeline has: (1) document loading, (2) text splitting, (3) embeddings, (4)\nvector store indexing, and (5) a retrieval + generation ch

# Embedding and vectorStoreDB

In [11]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any, Tuple
from sklearn.metrics.pairwise import  cosine_similarity



In [12]:
class EmbeddingManager:
    """ handles document embeddings generation using SentenceTransformer """

    def __init__(self, model_name:str = "all-MiniLM-L6-v2"):
        """ initialize the embedding manager """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """load sentence transformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")

            # This loads the neural network weights into the memory
            self.model = SentenceTransformer(self.model_name)

            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {self.model_name} : {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Converts english test chunks into list of numbers(vectors)
        Returns: A numPy array (matrix) where each row is different text chunk

        """
        #safety check to ensure model actually exists in ram
        if not self.model:
            raise ValueError("Nodel not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")

        # encode() is the math engine. It runs the text through the transformer layers.
        # show_progress_bar is perfect for tracking progress
        embeddings = self.model.encode(texts,show_progress_bar=True)

        # shape shows ( Number of chunks, Embedding dimension)
        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager





Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4824.38it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


# VectorStore

In [26]:
class VectorStore:
    """ Manages document embeddings in a chromaDB vector store """
    def __init__(self, collection_name:str="pdf_documents", persist_directory:str="../data/vector_store"):
        """
        Initialized the vector store
            :param collection_name: name of the ChromaDB collection
            :param persist_directory: Directory to persist the vecotr store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the chromaDB client and collection"""
        try:
            #create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Initialized collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error creating collection: {self.collection_name} : {e}")
            raise

    def add_document(self, documents: List[Any], embeddings: np.ndarray):
        """
        Adds documents ad their embeddings to the collection
            :param documents: List of langchain documents
            :param embeddings: Corresponding embeddings
        """
        if len(documents)!=len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding {len(documents)} documents to collection: {self.collection_name}")

        #Prepare data for chroma DB
        ids = []
        metadatas = []
        document_text = []
        embeddings_list = []

        for i , (doc,embedding) in enumerate(zip(documents,embeddings)):
            # Generate unique ids
            doc_id = f"{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare the metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            #document content
            document_text.append(doc.page_content)

            #embeddings
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids = ids,
                metadatas = metadatas,
                embeddings = embeddings_list,
                documents = document_text
            )
            print(f"Successfully added {len(ids)} documents to document vector store")
            print(f"Total documents in collection : {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to collection: {self.collection_name}")
            raise

vectorstore = VectorStore()
vectorstore

Initialized collection: pdf_documents
Existing documents in collection: 0


In [28]:
### Convert the text to embeddings
texts= [doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings = embedding_manager.generate_embeddings(texts)

## Store it in the vector database
vectorstore.add_document(chunks,embeddings)

Generating embeddings for 4 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.45it/s]

Generated embeddings with shape: (4, 384)
Adding 4 documents to collection: pdf_documents
Successfully added 4 documents to document vector store
Total documents in collection : 8
